# CardioScore Validation 05 — Publish Results to GitHub `main`

This notebook publishes **derived validation artifacts only**. It never uploads raw datasets, secrets, or arbitrary files. Results are copied to `validation_results/<RUN_LABEL>/` and committed to `main`.

**Security:** create a Colab Secret named `GITHUB_TOKEN` containing a GitHub token with permission to write to this repository. The token is read through Colab Secrets and is never written to disk or printed.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, tempfile, hashlib

REPO = "Virelion-Biotech/Virelion-CardioScore"
BRANCH = "main"
PIN = "869150cd5fb5ccf155fb066258404bd4df163ade"
LOCAL_RESULTS = Path("/content/cardioscore_validation/results")

assert LOCAL_RESULTS.is_dir(), f"Missing local results directory: {LOCAL_RESULTS}. Run notebooks 03/04 first."
print("Local result files:")
for p in sorted(LOCAL_RESULTS.rglob("*")):
    if p.is_file(): print(" ", p.relative_to(LOCAL_RESULTS))

In [ ]:
from google.colab import userdata

TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN and TOKEN.strip(), "Create a Colab Secret named GITHUB_TOKEN before publishing."
print("GitHub token loaded from Colab Secrets; value not displayed.")

In [ ]:
RUN_LABEL = input("Run label (example: patel_2019_2026-09-19): ").strip()
assert RUN_LABEL, "Run label cannot be empty."
assert all(c.isalnum() or c in "-_" for c in RUN_LABEL), "Run label may contain only letters, numbers, '-' and '_'."

allowed_names = {
    "locked_external_validation.json",
    "locked_compound_validation.csv",
    "locked_failures_by_compound.csv",
    "secondary_sensitivity.csv",
    "qc_log.csv",
    "qc_summary.json",
    "raw_mea_lineage.json",
    "source_receipt.json",
    "blinova_reference.csv",
    "blinova_semantic_summary.csv",
}
source_files = [p for p in LOCAL_RESULTS.rglob("*") if p.is_file()]
unexpected = [p.relative_to(LOCAL_RESULTS).as_posix() for p in source_files if p.name not in allowed_names]
assert not unexpected, f"Refusing to publish unapproved artifact(s): {unexpected}"
assert all(p.suffix.lower() in {".json", ".csv", ".md"} for p in source_files), "Only JSON/CSV/Markdown artifacts are publishable."
print("Approved artifacts:", len(source_files))

In [ ]:
work = Path(tempfile.mkdtemp(prefix="cardioscore-publish-"))
askpass = work / "askpass.sh"
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf \'%s\\n\' "x-access-token" ;;\n  *Password*) printf \'%s\\n\' "$GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n', encoding="utf-8")
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = TOKEN
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

repo_dir = work / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(repo_dir)], env=env, check=True)
print("Cloned main at:")
print(subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
dest = repo_dir / "validation_results" / RUN_LABEL
dest.mkdir(parents=True, exist_ok=False)
for src in source_files:
    target = dest / src.name
    shutil.copy2(src, target)

manifest = {
    "run_label": RUN_LABEL,
    "cardioscore_package_pin": PIN,
    "published_from": str(LOCAL_RESULTS),
    "artifacts": {},
}
for p in sorted(dest.iterdir()):
    if p.is_file():
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest["artifacts"][p.name] = {"bytes": p.stat().st_size, "sha256": h}
(dest / "PUBLISH_MANIFEST.json").write_text(json.dumps(manifest, indent=2)+"\n", encoding="utf-8")
print(json.dumps(manifest, indent=2))

In [ ]:
result = subprocess.run(["git", "-C", str(repo_dir), "status", "--short", "--untracked-files=all"], text=True, capture_output=True, check=True)
changed = [line[3:] if len(line) >= 4 else line for line in result.stdout.splitlines() if line.strip()]
expected_prefix = f"validation_results/{RUN_LABEL}/"
bad = [p for p in changed if not p.startswith(expected_prefix)]
assert not bad, f"Safety stop: git reports changes outside {expected_prefix}: {bad}"
assert changed, "No result files to commit."
print("Only expected validation artifacts changed:")
print(result.stdout)

In [ ]:
subprocess.run(["git", "-C", str(repo_dir), "add", f"validation_results/{RUN_LABEL}"], env=env, check=True)
subprocess.run(["git", "-C", str(repo_dir), "config", "user.name", "Syed Umer Hannan"], check=True)
subprocess.run(["git", "-C", str(repo_dir), "config", "user.email", "syedumerhannan@icloud.com"], check=True)

commit_message = f"Publish CardioScore validation results: {RUN_LABEL}"
subprocess.run(["git", "-C", str(repo_dir), "commit", "-m", commit_message], env=env, check=True)
local_commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()
print("Local commit:", local_commit)

subprocess.run(["git", "-C", str(repo_dir), "push", "origin", BRANCH], env=env, check=True)
remote = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "origin/main"], text=True).strip()
assert remote == local_commit, f"Push verification failed: origin/main={remote}, local={local_commit}"
print("Pushed and verified on main:", remote)

## What this publishes

The commit is restricted to the derived JSON/CSV/Markdown validation artifacts listed in the allowlist above. Raw `.zip`, `.mat`, `.h5/.hdf5`, `.csv` source datasets, credentials, and arbitrary files are intentionally excluded.

For the strongest audit trail, keep the `source_receipt.json`, `raw_mea_lineage.json`, and `PUBLISH_MANIFEST.json` with each run. They record hashes/provenance rather than embedding the source dataset.